#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Loading the Prepared Forecasting Datasets

In [2]:
processed_dir = Path("../data/processed")

daily_df = pd.read_csv(
    processed_dir / "daily_reservations.csv",
    parse_dates=["arrival_datetime"]
)

train_df = pd.read_csv(
    processed_dir / "train_daily.csv",
    parse_dates=["arrival_datetime"]
)

test_df = pd.read_csv(
    processed_dir / "test_daily.csv",
    parse_dates=["arrival_datetime"]
)

forecast_horizon = 7

print("Prepared Forecasting Datasets:")
print(f"Complete Dataset: {daily_df.shape}")
print(f"Training Dataset: {train_df.shape}")
print(f"Testing Dataset: {test_df.shape}")

print("\nForecasting Configuration:")
print(f"Target Variable: scheduled_reservations")
print(f"Forecast Horizon: {forecast_horizon} days")

print("\nTraining Period:")
print(f"{train_df['arrival_datetime'].min().date()} to {train_df['arrival_datetime'].max().date()}")

print("\nTesting Period:")
print(f"{test_df['arrival_datetime'].min().date()} to {test_df['arrival_datetime'].max().date()}")

print("\nDataset Preview:")
print(daily_df.head().to_string(index=False))

Prepared Forecasting Datasets:
Complete Dataset: (549, 2)
Training Dataset: (439, 2)
Testing Dataset: (110, 2)

Forecasting Configuration:
Target Variable: scheduled_reservations
Forecast Horizon: 7 days

Training Period:
2017-07-01 to 2018-09-12

Testing Period:
2018-09-13 to 2018-12-31

Dataset Preview:
arrival_datetime  scheduled_reservations
      2017-07-01                      68
      2017-07-02                       5
      2017-07-03                       3
      2017-07-04                       4
      2017-07-05                       3


#### Key Observations
- **Successful Dataset Loading:** The complete dataset contains 549 daily observations, with 439 training observations and 110 testing observations.
- **Forecasting Configuration Confirmed:** The target is `scheduled_reservations`, with a 7-day forecasting horizon and chronologically separated training and testing periods.

#### Validating the Forecasting Data Structure

In [3]:
print("Dataset Structure Validation:")

for name, dataset in [
    ("Complete", daily_df),
    ("Training", train_df),
    ("Testing", test_df)
]:
    print(f"\n{name} Dataset:")
    print(f"Missing Values: {dataset.isna().sum().sum()}")
    print(f"Duplicated Dates: {dataset['arrival_datetime'].duplicated().sum()}")
    print(f"Chronologically Sorted: {dataset['arrival_datetime'].is_monotonic_increasing}")
    print(f"Negative Target Values: {(dataset['scheduled_reservations'] < 0).sum()}")

print("\nChronological Split Validation:")

combined_df = pd.concat(
    [train_df, test_df],
    ignore_index=True
)

print(f"Combined Dataset Matches Original: {combined_df.equals(daily_df)}")

print(
    "Training Ends Before Testing:",
    train_df["arrival_datetime"].max()
    < test_df["arrival_datetime"].min()
)

print(
    "Consecutive Split Boundary:",
    train_df["arrival_datetime"].max() + pd.Timedelta(days=1)
    == test_df["arrival_datetime"].min()
)

Dataset Structure Validation:

Complete Dataset:
Missing Values: 0
Duplicated Dates: 0
Chronologically Sorted: True
Negative Target Values: 0

Training Dataset:
Missing Values: 0
Duplicated Dates: 0
Chronologically Sorted: True
Negative Target Values: 0

Testing Dataset:
Missing Values: 0
Duplicated Dates: 0
Chronologically Sorted: True
Negative Target Values: 0

Chronological Split Validation:
Combined Dataset Matches Original: True
Training Ends Before Testing: True
Consecutive Split Boundary: True


#### Key Observations
- **Data Integrity Confirmed:** All three datasets contain no missing values, duplicated dates, or negative reservation counts, and their observations are chronologically ordered.
- **Chronological Split Verified:** The training and testing datasets reconstruct the complete forecasting series exactly, with consecutive, nonoverlapping date boundaries.

#### Engineering Calendar-Based Features

In [4]:
feature_df = daily_df.copy()

feature_df["year"] = feature_df["arrival_datetime"].dt.year
feature_df["quarter"] = feature_df["arrival_datetime"].dt.quarter
feature_df["month"] = feature_df["arrival_datetime"].dt.month
feature_df["day_of_month"] = feature_df["arrival_datetime"].dt.day
feature_df["day_of_week"] = feature_df["arrival_datetime"].dt.dayofweek
feature_df["day_of_year"] = feature_df["arrival_datetime"].dt.dayofyear
feature_df["week_of_year"] = feature_df["arrival_datetime"].dt.isocalendar().week.astype(int)

feature_df["is_weekend"] = (
    feature_df["day_of_week"] >= 5
).astype(int)

feature_df["is_month_start"] = (
    feature_df["arrival_datetime"].dt.is_month_start
).astype(int)

feature_df["is_month_end"] = (
    feature_df["arrival_datetime"].dt.is_month_end
).astype(int)

print("Calendar Feature Engineering Summary:")
print(f"Original Features: {daily_df.shape[1]}")
print(f"Updated Features: {feature_df.shape[1]}")
print(f"New Calendar Features: {feature_df.shape[1] - daily_df.shape[1]}")

print("\nEngineered Calendar Features:")
print(feature_df.columns.drop(daily_df.columns).tolist())

print("\nFeature Preview:")
print(feature_df.head(10).to_string(index=False))

print("\nMissing Values:")
print(feature_df.isna().sum().sum())

Calendar Feature Engineering Summary:
Original Features: 2
Updated Features: 12
New Calendar Features: 10

Engineered Calendar Features:
['year', 'quarter', 'month', 'day_of_month', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end']

Feature Preview:
arrival_datetime  scheduled_reservations  year  quarter  month  day_of_month  day_of_week  day_of_year  week_of_year  is_weekend  is_month_start  is_month_end
      2017-07-01                      68  2017        3      7             1            5          182            26           1               1             0
      2017-07-02                       5  2017        3      7             2            6          183            26           1               0             0
      2017-07-03                       3  2017        3      7             3            0          184            27           0               0             0
      2017-07-04                       4  2017        3      7        

C:\Users\user\AppData\Local\Temp\ipykernel_5076\289446371.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  feature_df["day_of_week"] = feature_df["arrival_datetime"].dt.dayofweek
C:\Users\user\AppData\Local\Temp\ipykernel_5076\289446371.py:8: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  feature_df["day_of_year"] = feature_df["arrival_datetime"].dt.dayofyear


#### Key Observations
- **Successful Calendar Feature Engineering:** Ten calendar-based features were created, expanding the dataset from 2 to 12 columns without introducing missing values.
- **Forecasting Compatibility:** The engineered features capture calendar characteristics known in advance, making them suitable for the 7-day forecasting horizon.

#### Updating Deprecated Datetime Properties

In [5]:
original_day_of_week = feature_df["day_of_week"].copy()
original_day_of_year = feature_df["day_of_year"].copy()

feature_df["day_of_week"] = feature_df["arrival_datetime"].dt.day_of_week
feature_df["day_of_year"] = feature_df["arrival_datetime"].dt.day_of_year

print("Datetime Property Validation:")
print(f"Day of Week Values Unchanged: {feature_df['day_of_week'].equals(original_day_of_week)}")
print(f"Day of Year Values Unchanged: {feature_df['day_of_year'].equals(original_day_of_year)}")

print("\nDataset Validation:")
print(f"Dataset Shape: {feature_df.shape}")
print(f"Missing Values: {feature_df.isna().sum().sum()}")
print(f"Calendar Features: {feature_df.shape[1] - daily_df.shape[1]}")

Datetime Property Validation:
Day of Week Values Unchanged: True
Day of Year Values Unchanged: True

Dataset Validation:
Dataset Shape: (549, 12)
Missing Values: 0
Calendar Features: 10


#### Key Observations
- **Datetime Compatibility Confirmed:** The updated `day_of_week` and `day_of_year` properties produce identical values to their deprecated equivalents.
- **Feature Integrity Preserved:** All 10 calendar features remain intact across 549 observations, with no missing values.

#### Engineering Historical Lag Features

In [6]:
lag_periods = [7, 14, 21, 28]

for lag in lag_periods:
    feature_df[f"reservations_lag_{lag}"] = (
        feature_df["scheduled_reservations"].shift(lag)
    )

print("Historical Lag Feature Engineering Summary:")
print(f"Dataset Shape: {feature_df.shape}")
print(f"New Lag Features: {len(lag_periods)}")

print("\nEngineered Lag Features:")
print([f"reservations_lag_{lag}" for lag in lag_periods])

print("\nMissing Values by Lag Feature:")
print(
    feature_df[
        [f"reservations_lag_{lag}" for lag in lag_periods]
    ].isna().sum().to_string()
)

print("\nLag Feature Preview:")
print(
    feature_df[
        ["arrival_datetime", "scheduled_reservations"]
        + [f"reservations_lag_{lag}" for lag in lag_periods]
    ].iloc[28:38].to_string(index=False)
)

Historical Lag Feature Engineering Summary:
Dataset Shape: (549, 16)
New Lag Features: 4

Engineered Lag Features:
['reservations_lag_7', 'reservations_lag_14', 'reservations_lag_21', 'reservations_lag_28']

Missing Values by Lag Feature:
reservations_lag_7      7
reservations_lag_14    14
reservations_lag_21    21
reservations_lag_28    28

Lag Feature Preview:
arrival_datetime  scheduled_reservations  reservations_lag_7  reservations_lag_14  reservations_lag_21  reservations_lag_28
      2017-07-29                      11                 7.0                  6.0                  7.0                 68.0
      2017-07-30                       6                 4.0                  6.0                  4.0                  5.0
      2017-07-31                       7                 1.0                 56.0                  5.0                  3.0
      2017-08-01                      33                29.0                 26.0                 42.0                  4.0
      2017-08-0

#### Key Observations
- **Successful Lag Feature Engineering:** Four historical reservation features were created using 7-, 14-, 21-, and 28-day lags, expanding the dataset to 16 columns.
- **Expected Historical Gaps:** The first 7, 14, 21, and 28 observations contain missing values in their respective lag features because sufficient historical records are unavailable.
- **Forecasting Horizon Compatibility:** All selected lags use observations at least seven days before their target dates, supporting fixed-origin predictions across the 7-day forecasting horizon.

#### Engineering Rolling Statistical Features

In [7]:
rolling_windows = [7, 14, 28]

historical_reservations = (
    feature_df["scheduled_reservations"].shift(forecast_horizon)
)

for window in rolling_windows:
    feature_df[f"rolling_mean_{window}"] = (
        historical_reservations.rolling(window=window).mean()
    )

    feature_df[f"rolling_std_{window}"] = (
        historical_reservations.rolling(window=window).std()
    )

print("Rolling Feature Engineering Summary:")
print(f"Dataset Shape: {feature_df.shape}")
print(f"New Rolling Features: {len(rolling_windows) * 2}")

rolling_columns = [
    f"rolling_{stat}_{window}"
    for window in rolling_windows
    for stat in ["mean", "std"]
]

print("\nEngineered Rolling Features:")
print(rolling_columns)

print("\nMissing Values by Rolling Feature:")
print(feature_df[rolling_columns].isna().sum().to_string())

print("\nRolling Feature Preview:")
print(
    feature_df[
        ["arrival_datetime", "scheduled_reservations"] + rolling_columns
    ].iloc[34:44].round(2).to_string(index=False)
)

Rolling Feature Engineering Summary:
Dataset Shape: (549, 22)
New Rolling Features: 6

Engineered Rolling Features:
['rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_std_28']

Missing Values by Rolling Feature:
rolling_mean_7     13
rolling_std_7      13
rolling_mean_14    20
rolling_std_14     20
rolling_mean_28    34
rolling_std_28     34

Rolling Feature Preview:
arrival_datetime  scheduled_reservations  rolling_mean_7  rolling_std_7  rolling_mean_14  rolling_std_14  rolling_mean_28  rolling_std_28
      2017-08-04                       9            8.14           9.56            11.57           15.38            12.11           16.93
      2017-08-05                      51            8.71           9.60            11.93           15.29            10.07           12.91
      2017-08-06                      25            9.00           9.47            11.93           15.29            10.11           12.90
      2017-08-07            

C:\Users\user\AppData\Local\Temp\ipykernel_5076\1348827042.py:36: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].iloc[34:44].round(2).to_string(index=False)


#### Key Observations
- **Successful Rolling Feature Engineering:** Six features were created using 7-, 14-, and 28-day rolling means and standard deviations, expanding the dataset to 22 columns.
- **Expected Historical Gaps:** The rolling features contain 13, 20, and 34 initial missing values, respectively, reflecting the seven-day historical shift and the observations required for each window.
- **Forecasting Integrity:** All rolling statistics use reservation counts at least seven days before their corresponding target dates, avoiding direct use of observations within the fixed seven-day forecast window.

#### Validating Rolling Feature Values and Formatting

In [8]:
rolling_preview = feature_df[
    ["arrival_datetime", "scheduled_reservations"] + rolling_columns
].iloc[34:44].copy()

rolling_preview[rolling_columns] = (
    rolling_preview[rolling_columns].round(2)
)

print("Rolling Feature Preview:")
print(rolling_preview.to_string(index=False))

print("\nRolling Feature Validation:")
print(f"Rolling Features Created: {len(rolling_columns)}")
print(f"Dataset Shape: {feature_df.shape}")
print(f"Negative Rolling Means: {(feature_df[[f'rolling_mean_{w}' for w in rolling_windows]] < 0).sum().sum()}")
print(f"Negative Rolling Standard Deviations: {(feature_df[[f'rolling_std_{w}' for w in rolling_windows]] < 0).sum().sum()}")

print("\nHistorical Availability:")
for window in rolling_windows:
    expected_missing = forecast_horizon + window - 1
    actual_missing = feature_df[f"rolling_mean_{window}"].isna().sum()

    print(
        f"{window}-Day Window: "
        f"Expected Missing = {expected_missing}, "
        f"Actual Missing = {actual_missing}, "
        f"Valid = {expected_missing == actual_missing}"
    )

Rolling Feature Preview:
arrival_datetime  scheduled_reservations  rolling_mean_7  rolling_std_7  rolling_mean_14  rolling_std_14  rolling_mean_28  rolling_std_28
      2017-08-04                       9            8.14           9.56            11.57           15.38            12.11           16.93
      2017-08-05                      51            8.71           9.60            11.93           15.29            10.07           12.91
      2017-08-06                      25            9.00           9.47            11.93           15.29            10.11           12.90
      2017-08-07                      24            9.86           8.88             8.43            8.55            10.25           12.84
      2017-08-08                      48           10.43          10.33             8.93            9.78            11.29           13.47
      2017-08-09                      24           10.86          10.02             9.00            9.75            11.39           13.41
      201

#### Key Observations
- **Rolling Feature Integrity:** All six rolling statistical features were validated successfully, with no negative means or standard deviations.
- **Historical Availability Confirmed:** The initial missing-value counts match the expected requirements of each rolling window, confirming consistent application of the 7-day historical shift.

#### Engineering Expanding Statistical Features

In [9]:
historical_reservations = (
    feature_df["scheduled_reservations"].shift(forecast_horizon)
)

feature_df["expanding_mean"] = (
    historical_reservations.expanding(min_periods=7).mean()
)

feature_df["expanding_std"] = (
    historical_reservations.expanding(min_periods=7).std()
)

print("Expanding Feature Engineering Summary:")
print(f"Dataset Shape: {feature_df.shape}")
print("New Expanding Features: 2")

print("\nMissing Values:")
print(
    feature_df[
        ["expanding_mean", "expanding_std"]
    ].isna().sum().to_string()
)

print("\nExpanding Feature Preview:")
expanding_preview = feature_df[
    [
        "arrival_datetime",
        "scheduled_reservations",
        "expanding_mean",
        "expanding_std"
    ]
].iloc[28:38].copy()

expanding_preview[
    ["expanding_mean", "expanding_std"]
] = expanding_preview[
    ["expanding_mean", "expanding_std"]
].round(2)

print(expanding_preview.to_string(index=False))

print("\nExpanding Feature Validation:")
print(f"Negative Expanding Means: {(feature_df['expanding_mean'] < 0).sum()}")
print(f"Negative Expanding Standard Deviations: {(feature_df['expanding_std'] < 0).sum()}")

Expanding Feature Engineering Summary:
Dataset Shape: (549, 24)
New Expanding Features: 2

Missing Values:
expanding_mean    13
expanding_std     13

Expanding Feature Preview:
arrival_datetime  scheduled_reservations  expanding_mean  expanding_std
      2017-07-29                      11           13.14          18.36
      2017-07-30                       6           12.74          18.04
      2017-07-31                       7           12.25          17.81
      2017-08-01                      33           12.92          17.75
      2017-08-02                       6           12.54          17.50
      2017-08-03                       8           12.41          17.18
      2017-08-04                       9           12.11          16.93
      2017-08-05                      51           12.07          16.63
      2017-08-06                      25           11.87          16.37
      2017-08-07                      24           11.71          16.12

Expanding Feature Validation:


#### Key Observations
- **Successful Expanding Feature Engineering:** Two features were created to capture the cumulative historical mean and standard deviation of daily scheduled reservations.
- **Historical Integrity Confirmed:** Both features contain 13 expected initial missing values, with no negative means or standard deviations, and use reservation observations at least seven days before each target date.

#### Engineering Historical Trend Features

In [10]:
feature_df["trend_7_day"] = (
    feature_df["rolling_mean_7"]
    - feature_df["rolling_mean_7"].shift(7)
)

feature_df["trend_14_day"] = (
    feature_df["rolling_mean_14"]
    - feature_df["rolling_mean_14"].shift(14)
)

trend_columns = ["trend_7_day", "trend_14_day"]

print("Historical Trend Feature Engineering Summary:")
print(f"Dataset Shape: {feature_df.shape}")
print(f"New Trend Features: {len(trend_columns)}")

print("\nMissing Values:")
print(feature_df[trend_columns].isna().sum().to_string())

print("\nHistorical Trend Feature Preview:")
trend_preview = feature_df[
    ["arrival_datetime", "scheduled_reservations"] + trend_columns
].iloc[40:50].copy()

trend_preview[trend_columns] = (
    trend_preview[trend_columns].round(2)
)

print(trend_preview.to_string(index=False))

print("\nTrend Feature Summary:")
print(feature_df[trend_columns].describe().round(2).to_string())

Historical Trend Feature Engineering Summary:
Dataset Shape: (549, 26)
New Trend Features: 2

Missing Values:
trend_7_day     20
trend_14_day    34

Historical Trend Feature Preview:
arrival_datetime  scheduled_reservations  trend_7_day  trend_14_day
      2017-08-10                      42         3.00         -4.21
      2017-08-11                      48         3.29         -2.71
      2017-08-12                      32         8.43          0.43
      2017-08-13                      24        10.86          1.93
      2017-08-14                     191        12.43          3.86
      2017-08-15                      51        14.00          6.14
      2017-08-16                      17        16.14          7.57
      2017-08-17                      36        21.14          9.79
      2017-08-18                      35        26.00         12.86
      2017-08-19                      30        17.57         14.00

Trend Feature Summary:
       trend_7_day  trend_14_day
count       

#### Key Observations
- **Successful Trend Feature Engineering:** Two historical trend features were created, capturing changes between consecutive 7-day and 14-day rolling averages.
- **Historical Trend Variability:** The 7-day trend ranges from −64.14 to 46.43 reservations, while the 14-day trend ranges from −45.36 to 46.64, indicating both increasing and decreasing historical reservation activity.
- **Expected Historical Gaps:** The trend features contain 20 and 34 initial missing values, respectively, consistent with their historical window requirements.

#### Validating Feature Availability and Forecasting Integrity

In [11]:
target_column = "scheduled_reservations"
date_column = "arrival_datetime"

feature_columns = [
    col for col in feature_df.columns
    if col not in [date_column, target_column]
]

historical_columns = [
    col for col in feature_columns
    if col.startswith((
        "reservations_lag_",
        "rolling_",
        "expanding_",
        "trend_"
    ))
]

expected_missing = {
    "reservations_lag_7": 7,
    "reservations_lag_14": 14,
    "reservations_lag_21": 21,
    "reservations_lag_28": 28,
    "rolling_mean_7": 13,
    "rolling_std_7": 13,
    "rolling_mean_14": 20,
    "rolling_std_14": 20,
    "rolling_mean_28": 34,
    "rolling_std_28": 34,
    "expanding_mean": 13,
    "expanding_std": 13,
    "trend_7_day": 20,
    "trend_14_day": 34
}

missing_validation = {
    column: feature_df[column].isna().sum() == expected
    for column, expected in expected_missing.items()
}

complete_feature_mask = feature_df[feature_columns].notna().all(axis=1)

print("Feature Engineering Summary:")
print(f"Total Observations: {len(feature_df):,}")
print(f"Total Predictor Features: {len(feature_columns)}")
print(f"Calendar Features: {len(feature_columns) - len(historical_columns)}")
print(f"Historical Features: {len(historical_columns)}")

print("\nHistorical Feature Availability:")
print(f"Observations With Complete Features: {complete_feature_mask.sum():,}")
print(f"Initial Observations Without Complete Features: {(~complete_feature_mask).sum():,}")
print(
    "First Complete Feature Date:",
    feature_df.loc[complete_feature_mask, date_column].min().date()
)

print("\nMissing Value Validation:")
for column, valid in missing_validation.items():
    print(f"{column}: {valid}")

print("\nForecasting Integrity:")
print(f"Minimum Historical Lag: {min(lag_periods)} days")
print(f"Forecast Horizon: {forecast_horizon} days")
print(f"Historical Lag Meets Horizon: {min(lag_periods) >= forecast_horizon}")
print(f"All Missing Counts Valid: {all(missing_validation.values())}")
print(f"Missing Values After Initial Warm-Up: {feature_df.loc[complete_feature_mask.idxmax():, feature_columns].isna().sum().sum()}")

Feature Engineering Summary:
Total Observations: 549
Total Predictor Features: 24
Calendar Features: 10
Historical Features: 14

Historical Feature Availability:
Observations With Complete Features: 515
Initial Observations Without Complete Features: 34
First Complete Feature Date: 2017-08-04

Missing Value Validation:
reservations_lag_7: True
reservations_lag_14: True
reservations_lag_21: True
reservations_lag_28: True
rolling_mean_7: True
rolling_std_7: True
rolling_mean_14: True
rolling_std_14: True
rolling_mean_28: True
rolling_std_28: True
expanding_mean: True
expanding_std: True
trend_7_day: True
trend_14_day: True

Forecasting Integrity:
Minimum Historical Lag: 7 days
Forecast Horizon: 7 days
Historical Lag Meets Horizon: True
All Missing Counts Valid: True
Missing Values After Initial Warm-Up: 0


#### Key Observations
- **Feature Engineering Completed:** The dataset contains 24 predictor features, comprising 10 calendar-based and 14 historical features.
- **Historical Warm-Up Identified:** The first 34 observations lack sufficient historical information, leaving 515 complete observations beginning August 4, 2017.
- **Feature Availability Validated:** All 14 historical features have the expected missing-value patterns, with no missing predictor values after the initial warm-up period.

#### Preparing Feature-Engineered Training and Testing Datasets

In [12]:
modeling_df = (
    feature_df.dropna(subset=feature_columns)
    .copy()
    .reset_index(drop=True)
)

train_end_date = train_df["arrival_datetime"].max()
test_start_date = test_df["arrival_datetime"].min()

train_features_df = (
    modeling_df[
        modeling_df["arrival_datetime"] <= train_end_date
    ]
    .copy()
    .reset_index(drop=True)
)

test_features_df = (
    modeling_df[
        modeling_df["arrival_datetime"] >= test_start_date
    ]
    .copy()
    .reset_index(drop=True)
)

print("Feature-Engineered Dataset Summary:")
print(f"Complete Modeling Dataset: {modeling_df.shape}")
print(f"Training Dataset: {train_features_df.shape}")
print(f"Testing Dataset: {test_features_df.shape}")

print("\nTraining Period:")
print(f"Start Date: {train_features_df['arrival_datetime'].min().date()}")
print(f"End Date: {train_features_df['arrival_datetime'].max().date()}")

print("\nTesting Period:")
print(f"Start Date: {test_features_df['arrival_datetime'].min().date()}")
print(f"End Date: {test_features_df['arrival_datetime'].max().date()}")

print("\nDataset Validation:")
print(
    "All Modeling Observations Preserved:",
    len(train_features_df) + len(test_features_df) == len(modeling_df)
)

print(
    "Chronological Separation:",
    train_features_df["arrival_datetime"].max()
    < test_features_df["arrival_datetime"].min()
)

print(
    "Missing Predictor Values:",
    modeling_df[feature_columns].isna().sum().sum()
)

print(
    "Test Period Preserved:",
    len(test_features_df) == len(test_df)
)

Feature-Engineered Dataset Summary:
Complete Modeling Dataset: (515, 26)
Training Dataset: (405, 26)
Testing Dataset: (110, 26)

Training Period:
Start Date: 2017-08-04
End Date: 2018-09-12

Testing Period:
Start Date: 2018-09-13
End Date: 2018-12-31

Dataset Validation:
All Modeling Observations Preserved: True
Chronological Separation: True
Missing Predictor Values: 0
Test Period Preserved: True


#### Key Observations
- **Successful Modeling Dataset Preparation:** Removing the 34 initial warm-up observations produced 515 complete observations, containing 24 predictor features and the target variable.
- **Chronological Split Preserved:** The training dataset contains 405 observations, while the testing dataset retains all 110 observations without overlapping dates.
- **Feature Completeness Confirmed:** All modeling observations are preserved across the two datasets, with no missing predictor values.

#### Validating Feature-Engineered Dataset Integrity

In [13]:
original_train_targets = train_df.set_index("arrival_datetime")[
    "scheduled_reservations"
]

original_test_targets = test_df.set_index("arrival_datetime")[
    "scheduled_reservations"
]

engineered_train_targets = train_features_df.set_index("arrival_datetime")[
    "scheduled_reservations"
]

engineered_test_targets = test_features_df.set_index("arrival_datetime")[
    "scheduled_reservations"
]

validation_results = {
    "Training Targets Preserved": engineered_train_targets.equals(
        original_train_targets.loc[engineered_train_targets.index]
    ),
    "Testing Targets Preserved": engineered_test_targets.equals(
        original_test_targets
    ),
    "Training Dates Sorted": train_features_df[
        "arrival_datetime"
    ].is_monotonic_increasing,
    "Testing Dates Sorted": test_features_df[
        "arrival_datetime"
    ].is_monotonic_increasing,
    "No Overlapping Dates": (
        train_features_df["arrival_datetime"].max()
        < test_features_df["arrival_datetime"].min()
    ),
    "Feature Columns Consistent": (
        train_features_df.columns.equals(test_features_df.columns)
    ),
    "No Missing Predictor Values": (
        modeling_df[feature_columns].notna().all().all()
    ),
    "No Missing Target Values": (
        modeling_df[target_column].notna().all()
    ),
    "All Test Observations Retained": (
        len(test_features_df) == len(test_df)
    )
}

print("Feature-Engineered Dataset Integrity:")
for check, result in validation_results.items():
    print(f"{check}: {result}")

print("\nOverall Validation:")
print(f"All Checks Passed: {all(validation_results.values())}")

Feature-Engineered Dataset Integrity:
Training Targets Preserved: True
Testing Targets Preserved: True
Training Dates Sorted: True
Testing Dates Sorted: True
No Overlapping Dates: True
Feature Columns Consistent: True
No Missing Predictor Values: True
No Missing Target Values: True
All Test Observations Retained: True

Overall Validation:
All Checks Passed: True


#### Key Observations
- **Target Integrity Preserved:** Feature engineering has not altered the original reservation counts, and all 110 testing observations remain intact.
- **Modeling Dataset Validated:** Training and testing datasets have identical feature structures, no missing values, and correctly ordered, nonoverlapping dates.

#### Saving Feature-Engineered Forecasting Datasets

In [15]:
feature_files = {
    "feature_engineered_daily.csv": modeling_df,
    "train_features.csv": train_features_df,
    "test_features.csv": test_features_df
}

print("Saved Feature-Engineered Dataset Validation:")

for filename, dataset in feature_files.items():
    file_path = processed_dir / filename

    dataset.to_csv(file_path, index=False)

    reloaded_df = pd.read_csv(
        file_path,
        parse_dates=["arrival_datetime"]
    )

    pd.testing.assert_frame_equal(
        dataset.reset_index(drop=True),
        reloaded_df,
        check_dtype=False,
        check_exact=False,
        rtol=1e-12,
        atol=1e-12
    )

    print(f"\nArtifact: {filename}")
    print(f"File Exists: {file_path.exists()}")
    print(f"Dataset Shape: {reloaded_df.shape}")
    print("Data Values Preserved: True")
    print(f"Missing Values: {reloaded_df.isna().sum().sum()}")

print("\nFinal Artifact Summary:")
print(f"Total Artifacts Validated: {len(feature_files)}")
print(f"Predictor Features: {len(feature_columns)}")
print(f"Forecast Horizon: {forecast_horizon} days")

Saved Feature-Engineered Dataset Validation:

Artifact: feature_engineered_daily.csv
File Exists: True
Dataset Shape: (515, 26)
Data Values Preserved: True
Missing Values: 0

Artifact: train_features.csv
File Exists: True
Dataset Shape: (405, 26)
Data Values Preserved: True
Missing Values: 0

Artifact: test_features.csv
File Exists: True
Dataset Shape: (110, 26)
Data Values Preserved: True
Missing Values: 0

Final Artifact Summary:
Total Artifacts Validated: 3
Predictor Features: 24
Forecast Horizon: 7 days


#### Key Observations
- **Successful Artifact Preservation:** All three feature-engineered datasets were saved and reloaded successfully, preserving their structure, values, and chronological ordering.
- **Modeling Readiness Confirmed:** The final datasets contain 24 predictor features with no missing values, comprising 405 training observations and 110 testing observations.

## Feature Engineering Summary

The feature engineering process transformed the prepared daily hotel reservation series into a structured forecasting dataset containing calendar-based and historical predictors.

The workflow developed and validated **24 predictor features** across 549 original daily observations, with 515 complete observations retained after accounting for historical feature availability.

### Calendar-Based Features

Ten calendar features were created to capture temporal patterns in scheduled hotel reservation activity.

These features include:

- **Year and Quarter:** Represent annual and quarterly calendar information.
- **Month and Day of Month:** Capture monthly and within-month calendar patterns.
- **Day of Week:** Represents recurring weekly patterns.
- **Day of Year and Week of Year:** Provide additional information about the position of each date within the calendar year.
- **Weekend Indicator:** Distinguishes weekends from weekdays.
- **Month Start and Month End Indicators:** Identify calendar boundaries.

Calendar features are known in advance and can therefore be used throughout the seven-day forecasting horizon.

### Historical Lag Features

Four lag features were created to represent historical reservation counts:

- `reservations_lag_7`
- `reservations_lag_14`
- `reservations_lag_21`
- `reservations_lag_28`

These features provide historical observations at weekly intervals, allowing forecasting models to learn relationships between current reservation volumes and earlier activity.

### Rolling Statistical Features

Six rolling statistical features were developed using historical windows of 7, 14, and 28 days.

For each window, the following statistics were calculated:

- **Rolling Mean:** Represents the average historical reservation volume.
- **Rolling Standard Deviation:** Measures variability in historical reservation activity.

A seven-day shift was applied before calculating these statistics to maintain compatibility with the forecasting horizon.

### Expanding Statistical Features

Two expanding features were created:

- `expanding_mean`
- `expanding_std`

These features summarize cumulative historical reservation levels and variability, providing longer-term context alongside shorter rolling windows.

### Historical Trend Features

Two historical trend indicators were developed:

- `trend_7_day`
- `trend_14_day`

These features measure changes between consecutive historical rolling averages, helping represent increases and decreases in reservation activity.

### Feature Availability and Validation

Historical lag, rolling, expanding, and trend calculations introduced expected missing values at the beginning of the series.

The first 34 observations were excluded because sufficient historical information was unavailable.

The resulting modeling dataset contains:

| Dataset | Observations | Predictor Features |
|---|---:|---:|
| Complete Feature-Engineered Dataset | 515 | 24 |
| Training Dataset | 405 | 24 |
| Testing Dataset | 110 | 24 |

Validation confirmed that:

- All engineered features follow the expected historical availability requirements.
- No predictor or target values are missing after the initial warm-up period.
- Original reservation counts remain unchanged.
- Training and testing datasets preserve their chronological boundaries.
- All 110 testing observations remain available for evaluation.
- All feature-engineered datasets were saved and reloaded successfully.

### Prepared Artifacts

The following reusable datasets were saved in `data/processed/`:

| Artifact | Description |
|---|---|
| `feature_engineered_daily.csv` | Complete feature-engineered forecasting dataset |
| `train_features.csv` | Feature-engineered chronological training dataset |
| `test_features.csv` | Feature-engineered chronological testing dataset |

### Forecasting Integrity

Historical features were constructed using observations at least seven days before their corresponding target dates, supporting the intended seven-day fixed-origin forecasting horizon.

However, feature construction alone does not guarantee leakage-free evaluation. Model development and evaluation must enforce forecast-origin boundaries, use time-aware validation, and ensure that historical target observations are available before being used as predictors.

Additionally, because the dataset is derived from retrospective scheduled-arrival records rather than historical booking snapshots, the forecasting exercise will be interpreted as retrospective time-series modeling rather than verified real-time operational forecasting.

**The completed feature engineering process establishes a validated, reusable, and temporally structured foundation for developing and evaluating daily hotel reservation forecasting models.**